<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_04_3_dropout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>


# T81-558: Anwendungen tiefer neuronaler Netzwerke

**Modul 4: Training für tabellarische Daten**

- Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
– Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).


# Modul 4 Material

- Teil 4.1: Verwenden der K-fachen Kreuzvalidierung mit PyTorch [[Video]](https://www.youtube.com/watch?v=Q8ZQNvZwsNE&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=Q8ZQNvZwsNE&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.2: Trainingspläne für PyTorch [[Video]](https://www.youtube.com/watch?v=lMMlbmfvKDQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=lMMlbmfvKDQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- **Teil 4.3: Dropout-Regularisierung** [[Video]](https://www.youtube.com/watch?v=4ixjgw6Q42U&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=4ixjgw6Q42U&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.4: Batch-Normalisierung [[Video]](https://www.youtube.com/watch?v=1U5nOKh9OLQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=1U5nOKh9OLQ&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
- Teil 4.5: RAPIDS für tabellarische Daten [[Video]](https://www.youtube.com/watch?v=KgoXuhG_kfs&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=KgoXuhG_kfs&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)


# Google CoLab-Anweisungen

Der folgende Code stellt sicher, dass Google CoLab ausgeführt wird und ordnet bei Bedarf Google Drive zu. Wir initialisieren das PyTorch-Gerät auch entweder auf GPU/MPS (falls verfügbar) oder CPU.


In [1]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import copy
import torch

try:
    import google.colab

    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Nutzen Sie eine GPU oder MPS (Apple), sofern verfügbar. (siehe Modul 3.2)
import torch

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")


# Frühzeitiges Stoppen (siehe Modul 3.4)
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_model = copy.deepcopy(model.state_dict())
            self.best_loss = val_loss
            self.counter = 0
            self.status = f"Improvement found, counter reset to {self.counter}"
        else:
            self.counter += 1
            self.status = f"No improvement in the last {self.counter} epochs"
            if self.counter >= self.patience:
                self.status = f"Early stopping triggered after {self.counter} epochs."
                if self.restore_best_weights:
                    model.load_state_dict(self.best_model)
                return True
        return False

Note: not using Google CoLab
Using device: mps


# Teil 4.3: Dropout-Regularisierung

Beim Erstellen effektiver Deep-Learning-Modelle stoßen wir häufig auf die Herausforderung der Überanpassung, bei der ein Modell mit Trainingsdaten außergewöhnlich gut funktioniert, sich jedoch nicht gut auf unbekannte Daten verallgemeinern lässt. Regularisierungstechniken sind daher von entscheidender Bedeutung, um sicherzustellen, dass unser Modell nicht unter dieser häufigen Falle leidet und ein Gleichgewicht zwischen Verzerrung (Unteranpassung) und Varianz (Überanpassung) herstellt. Regularisierungsmethoden funktionieren, indem sie die Komplexität des Modells bestrafen. Dadurch wird effektiv verhindert, dass das Modell zu viel Rauschen aus den Trainingsdaten lernt, und so seine Fähigkeit zur Generalisierung verbessert.

Eine solche leistungsstarke Regularisierungstechnik ist „Dropout“, ein Konzept, bei dem während des Trainings metaphorisch ein Teil der Neuronen im Modell „ausfällt“ oder vorübergehend „abgeschaltet“ wird, wodurch die gegenseitigen Abhängigkeiten der Neuronen verringert werden. Die durch das Ausscheiden von Neuronen eingeführte Zufälligkeit zwingt das Netzwerk, robustere Merkmale zu erlernen, was zu einem allgemeineren und weniger überangepassten Modell führt.

Dropout funktioniert anders als die meisten anderen Regularisierungstechniken. Anstatt der Verlustfunktion eine Strafe hinzuzufügen, deaktiviert es zufällig einen Teil der Neuronen (definiert durch einen Wahrscheinlichkeitsparameter, der normalerweise zwischen 0,2 und 0,5 liegt), wodurch für jede Trainingsinstanz effektiv eine andere Architektur des Netzwerks erstellt wird. Dies kann als Training eines Ensembles von Netzwerken betrachtet werden, was zu einem robusteren und verallgemeinerbareren Endmodell führt.

In diesem Abschnitt werden wir uns mit den Nuancen von Dropout befassen, seine theoretischen Grundlagen untersuchen und zeigen, wie es mithilfe verschiedener Deep-Learning-Frameworks angewendet werden kann. Das Verständnis und die korrekte Implementierung von Dropout ist ein wertvolles Werkzeug im Toolkit des Deep-Learning-Praktikers und unterstützt die Erstellung robuster und verallgemeinerter Modelle.

Im Verlauf des Kapitels werden Sie verstehen, wie Dropout perfekt in das Gesamtschema der Regularisierungsmethoden passt, Sie werden seine Vorteile und Grenzen kennenlernen und begreifen, wie Sie Dropout effektiv in Ihren Deep-Learning-Modellen einsetzen können.

Dropout ist eine Regularisierungstechnik, die von Geoffrey Hinton, einem Pionier auf dem Gebiet des Deep Learning, und seinen Studenten Nitish Srivastava, Geoffrey Hinton, Alex Krizhevsky, Ilya Sutskever und Ruslan Salakhutdinov in einem 2014 veröffentlichten Artikel mit dem Titel „Dropout: A Simple Way to Prevent Neural Networks from Overfitting“ vorgestellt wurde.

Die Entwicklung von Dropout erfolgte aufgrund der Erkenntnis, dass Überanpassung in großen neuronalen Netzwerken Probleme bereitet. Große neuronale Netzwerke mit Millionen von Parametern neigen aufgrund ihrer Fähigkeit, Trainingsdaten zu speichern, zu Überanpassung. Dies gilt insbesondere dann, wenn die Menge der verfügbaren Trainingsdaten im Verhältnis zur Komplexität des Netzwerks gering ist.

Geoffrey Hinton, der oft als „Pate des Deep Learning“ bezeichnet wird, hat mehrere bahnbrechende Beiträge zum Bereich der künstlichen Intelligenz geleistet, von denen Dropout nur einer ist. Er und sein Team suchten nach einfachen und effektiven Möglichkeiten, neuronale Netzwerke robuster zu machen und ihre Generalisierungsfähigkeiten zu verbessern. Inspiriert von biologischen Systemen schlugen sie die Idee des Dropouts vor, bei dem während des Trainingsprozesses eine Teilmenge von Neuronen zufällig „ausfällt“ oder deaktiviert wird, um zu verhindern, dass sie sich zu stark an die Daten anpassen.

Diese einfache, aber effektive Technik hat seitdem in der Deep-Learning-Community breite Anwendung gefunden und bildet die Grundlage für zahlreiche nachfolgende Forschungsarbeiten und Entwicklungen. Dropout hat sich als leistungsstarkes Tool beim Training neuronaler Netzwerke erwiesen, das dabei hilft, Überanpassung zu vermeiden und die Modellgeneralisierung zu verbessern, insbesondere in Szenarien mit begrenzten Trainingsdaten.

Wir beginnen damit, das vorherige Beispiel so zu ändern, dass Dropout verwendet wird. Wir werden die Daten auf die gleiche Weise wie zuvor vorverarbeiten.


In [2]:
import pandas as pd
from scipy.stats import zscore
from sklearn.model_selection import train_test_split

# Lesen des Datensatzes
df = read_csv(
    "https://data.heatonresearch.com/data/t81-558/jh-simple-dataset.csv",
    na_values=["NA", "?"],
)

# Dummies für den Job generieren
df = concat([df, get_dummies(df["job"], prefix="job", dtype=int)], axis=1)
df.drop("job", axis=1, inplace=True)

# Dummies für den Bereich generieren
df = concat([df, get_dummies(df["area"], prefix="area", dtype=int)], axis=1)
df.drop("area", axis=1, inplace=True)

# Dummies für Produkte generieren
df = concat([df, get_dummies(df["product"], prefix="product", dtype=int)], axis=1)
df.drop("product", axis=1, inplace=True)

# Fehlende Werte für Einkommen
med = df["income"].median()
df["income"] = df["income"].fillna(med)

# Bereiche standardisieren
df["income"] = zscore(df["income"])
df["aspect"] = zscore(df["aspect"])
df["save_rate"] = zscore(df["save_rate"])
df["subscriptions"] = zscore(df["subscriptions"])

Um Dropout zum bestehenden Code hinzuzufügen, verwende ich das Modul **nn.Dropout** von PyTorch. Es setzt während des Trainings zufällig einige Elemente des Eingabetensors mit der Wahrscheinlichkeit p (im Argument angegeben) auf Null, was dabei helfen kann, Überanpassung zu verhindern. Der geänderte Code wird unten mit den hinzugefügten Dropout-Ebenen angezeigt.

```
# Create the model with dropout
model = nn.Sequential(
    nn.Linear(x.shape[1], 20),
    nn.Dropout(0.5),
    nn.ReLU(),
    nn.Linear(20, 10),
    nn.Dropout(0.5),
    nn.ReLU(),
    nn.Linear(10, 1)
)
```

Die vollständigen Änderungen können Sie hier einsehen.

In [3]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# In PyTorch-Tensoren konvertieren
x_columns = df.columns.drop(["age", "id"])
x = torch.tensor(df[x_columns].values, dtype=torch.float32, device=device)
y = torch.tensor(df["age"].values, dtype=torch.float32, device=device).view(-1, 1)

# Legen Sie einen Zufallsstartwert für die Reproduzierbarkeit fest
torch.manual_seed(42)

# Kreuzvalidierung
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Parameter für vorzeitiges Stoppen
patience = 10

fold = 0
for train_idx, test_idx in kf.split(x):
    fold += 1
    print(f"Fold # {falten}")

    x_train, x_test = x[train_idx], x[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # PyTorch DataLoader
    train_dataset = TensorDataset(x_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Erstellen Sie das Modell mit Dropout
    model = nn.Sequential(
        nn.Linear(x.shape[1], 20),
        nn.Dropout(0.1),
        nn.ReLU(),
        nn.Linear(20, 10),
        nn.Dropout(0.1),
        nn.ReLU(),
        nn.Linear(10, 1),
    )
    model = torch.compile(model, backend="aot_eager").to(device)

    # Erstellen des Optimierers
    optimizer = optim.Adam(model.parameters())
    loss_fn = nn.MSELoss()

    # Variablen für frühzeitiges Stoppen
    best_loss = float("inf")
    early_stopping_counter = 0

    # Trainingsschleife
    EPOCHS = 500
    epoch = 0
    done = False
    es = EarlyStopping()

    while not done and epoch < EPOCHS:
        epoch += 1
        model.train()
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(x_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            optimizer.step()

        # Validierung
        model.eval()
        with torch.no_grad():
            val_output = model(x_test)
            val_loss = loss_fn(val_output, y_test)

        if es(model, val_loss):
            done = True

    print(
        f"Epoch {epoch}/{EPOCHS}, Validation Loss: " f"{val_loss.item()}, {es.status}"
    )

# Abschließende Bewertung
model.eval()
with torch.no_grad():
    oos_pred = model(x_test)
score = torch.sqrt(loss_fn(oos_pred, y_test)).item()
print(f"Faltungsbewertung (RMSE): {score}")

Fold #1


/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/autograd/function.py:539: UserWarning: The operator 'aten::native_dropout' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1702400227158/work/aten/src/ATen/mps/MPSFallback.mm:13.)
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/autograd/__init__.py:394: UserWarning: Error detected in LinearBackward0. Traceback of forward call that caused the error:
  File "/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/nn/modules/container.py", line 215, in forward
    input = module(input)
 (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1702400227158/work/torch/csrc/autograd/python_anomaly_mode.cpp:119.)
  result = Variable._execution_engine.run_backward

Epoch 95/500, Validation Loss: 4.972429275512695, Early stopping triggered after 5 epochs.
Fold #2
Epoch 51/500, Validation Loss: 19.33069610595703, Early stopping triggered after 5 epochs.
Fold #3
Epoch 50/500, Validation Loss: 13.100104331970215, Early stopping triggered after 5 epochs.
Fold #4
Epoch 51/500, Validation Loss: 13.250371932983398, Early stopping triggered after 5 epochs.
Fold #5
Epoch 42/500, Validation Loss: 15.019149780273438, Early stopping triggered after 5 epochs.
Fold score (RMSE): 3.725794792175293


Die Änderungen am Code sind unkompliziert, wir fügen einfach Dropout-Ebenen in die Architektur unseres Modells ein. Hier haben wir zwei Dropout-Ebenen hinzugefügt, eine zwischen der ersten und der zweiten linearen Ebene und eine weitere zwischen der zweiten und der dritten linearen Ebene. Jede Dropout-Ebene setzt während des Trainings zufällig 50 % der Eingabeelemente auf Null. Dropout-Ebenen werden normalerweise nach nichtlinearen Aktivierungsfunktionen hinzugefügt, wie in diesem Fall ReLU.

Dropout ist eine Regularisierungstechnik, die Überanpassung verhindert, indem sie das voneinander abhängige Lernen zwischen den Neuronen reduziert und die einzelnen Neuronen ermutigt, unabhängig zu sein. Während des Trainings deaktiviert Dropout zufällig einige Neuronen, wodurch die Daten gezwungen werden, neue Pfade zu finden, um sich durch das Netzwerk zu verbreiten. Dies führt folglich zu einem Netzwerk, das besser verallgemeinern kann und weniger wahrscheinlich die Trainingsdaten überanpasst.